# FlowGuard vision on a Colab GPU

Runs the FlowGuard camera pipeline (YOLO detector, ByteTrack tracker, then per-second counting) on a video clip and
produces the same `Measurement` JSONL the rest of FlowGuard reads.

You need four files on your computer:

1. `flowguard_vision_bundle.zip` (made by `uv run python scripts/pack_for_colab.py`)
2. a video clip of one approach from a fixed camera
3. one still frame from that clip (the reference frame)
4. the calibration file for that camera, drawn with `packages/vision/tools/calibrate.html`

Use **Runtime > Change runtime type > GPU** first. Then run the cells from top to bottom.

In [ ]:
!nvidia-smi -L
!pip -q install ultralytics opencv-python-headless pydantic

## 1. Upload the FlowGuard code

In [ ]:
import sys, zipfile
from pathlib import Path
from google.colab import files

up = files.upload()  # choose flowguard_vision_bundle.zip
bundle = next(n for n in up if n.endswith(".zip"))
zipfile.ZipFile(bundle).extractall("/content/flowguard")
sys.path.insert(0, "/content/flowguard")
import flowguard_contracts, flowguard_vision  # noqa: F401
print("FlowGuard code loaded")

## 2. Upload the clip, the reference frame and the calibration file

The calibration file names the reference frame, and the pipeline looks for that frame **in the same folder** as the
calibration file. Upload all three together.

In [ ]:
Path("/content/clip").mkdir(exist_ok=True)
up = files.upload()
for name, data in up.items():
    Path("/content/clip", name).write_bytes(data)
names = sorted(up)
VIDEO = next(Path("/content/clip", n) for n in names if n.lower().endswith((".mp4", ".avi", ".mov", ".mkv")))
CALIB = next(Path("/content/clip", n) for n in names if n.lower().endswith(".json"))
print("video:", VIDEO.name, "| calibration:", CALIB.name)

## 3. Run the pipeline

`MODEL` is the YOLO weights. `yolov8n.pt` is the small standard model, which only knows car, motorcycle, bus and truck
(no rickshaw or qingqi). To use your own fine-tuned weights, upload the `.pt` file and set `MODEL` to its path, and edit
`CLASSES` so each label of your model maps to one of the six FlowGuard classes.

In [ ]:
import time
from flowguard_vision.calibration import load_calibration
from flowguard_vision.run import measurements, write_jsonl
from flowguard_vision.video import frames_from_video, reference_frame
from flowguard_vision.yolo import COCO, YoloByteTrack

MODEL = "yolov8n.pt"
CLASSES = COCO            # {model label: FlowGuard class}
EVERY_S = 0.1             # process a frame every 0.1 s (10 frames per second)
CONFIDENCE = 0.25

calib = load_calibration(CALIB)
ref = reference_frame(CALIB, calib.reference_frame)
tracker = YoloByteTrack(MODEL, confidence=CONFIDENCE, classes=CLASSES)

t0 = time.time()
OUT = Path("/content/measurements.jsonl")
write_jsonl(measurements(frames_from_video(VIDEO, EVERY_S), tracker, calib, ref), OUT)
secs = sum(1 for _ in OUT.open())
print(f"{secs} seconds of footage in {time.time() - t0:.0f} s of compute -> {OUT}")

## 4. Look at the counts

In [ ]:
import json
import matplotlib.pyplot as plt

rows = [json.loads(line) for line in OUT.open()]
t = [r["time"] for r in rows]
classes = list(rows[0]["zone_counts"])
fig, ax = plt.subplots(3, 1, figsize=(11, 8), sharex=True)
for c in classes:
    ax[0].plot(t, [r["zone_counts"][c] for r in rows], label=c)
    ax[1].plot(t, [r["queue_counts"][c] for r in rows], label=c)
ax[2].plot(t, [sum(r["arrivals"].values()) if isinstance(r["arrivals"], dict) else r["arrivals"] for r in rows], label="arrivals")
ax[0].set_title("Vehicles in the zone"); ax[1].set_title("Vehicles waiting (stopped)"); ax[2].set_title("Arrivals per second")
ax[0].legend(ncol=3, fontsize=8); ax[2].set_xlabel("seconds into the clip")
plt.tight_layout(); plt.show()
print(json.dumps(rows[len(rows) // 2], indent=1)[:1500])

## 5. Check against the picture

The numbers above are only useful if the zone sits on the right road. This draws the zone, the stop line and the
detector boxes on a few frames. Look for vehicles that are missed, counted twice, or boxed outside the road.

In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO

model = YOLO(MODEL)
cap = cv2.VideoCapture(str(VIDEO))
fps = cap.get(cv2.CAP_PROP_FPS) or 25
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
picks = np.linspace(0, max(total - 1, 0), 6).astype(int)
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for a, idx in zip(axes.ravel(), picks):
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
    ok, frame = cap.read()
    if not ok:
        continue
    res = model.predict(frame, conf=CONFIDENCE, verbose=False)[0]
    img = res.plot()
    cv2.polylines(img, [np.array(calib.zone_polygon, np.int32)], True, (255, 120, 0), 2)
    cv2.line(img, tuple(int(v) for v in calib.stop_line[0]), tuple(int(v) for v in calib.stop_line[1]), (0, 0, 255), 3)
    a.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); a.set_title(f"t = {idx / fps:.1f} s"); a.axis("off")
cap.release(); plt.tight_layout(); plt.show()

## 6. Download the result

In [ ]:
files.download(str(OUT))